# Candidate generation and explainable selection

In [ ]:
#| default_exp design

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass
import math
from typing import Iterable, Mapping

from probeharbour.models import Candidate, Provenance
from probeharbour.sequence import normalize_sequence

In [ ]:
#| export
@dataclass(frozen=True)
class SelectionDecision:
    candidate_id: str
    selected: bool
    rank: int | None
    reasons: tuple[str, ...]

In [ ]:
#| export
def generate_junction_candidates(sequence: str, *, junction_position: int, arm_length: int, slides: Iterable[int],
                                 provenance: Provenance, candidate_prefix: str, junction_id: str | None = None) -> list[Candidate]:
    """Generate every in-bounds candidate around a transcript-coordinate junction."""
    transcript = normalize_sequence(sequence)
    result = []
    for slide in slides:
        cut = junction_position + int(slide)
        if cut - arm_length < 0 or cut + arm_length > len(transcript):
            continue
        result.append(Candidate(
            candidate_id=f"{candidate_prefix}:slide:{slide:+d}", rbd5=transcript[cut-arm_length:cut], rbd3=transcript[cut:cut+arm_length],
            provenance=provenance, junction_id=junction_id, ligation_site=cut, shift_nt=int(slide),
            on_junction=cut == junction_position, junction_spanning=True,
        ))
    return result

In [ ]:
#| export
def _rank_value(value: object, *, missing: float = -1.0) -> float:
    """Return a finite ranking value for optional numeric evidence."""
    try:
        result = float(value)
    except (TypeError, ValueError):
        return missing
    return result if math.isfinite(result) else missing

def select_candidates(rows: Iterable[Mapping[str, object]], *, limit: int = 1) -> list[SelectionDecision]:
    """Select by score then specificity; never hide why any row lost."""
    items = list(rows)
    ordered = sorted(items, key=lambda row: (
        -_rank_value(row.get("recalculated_score")),
        -_rank_value(row.get("specificity_percent")),
        str(row["candidate_id"]),
    ))
    decisions = []
    for index, row in enumerate(ordered):
        reasons = []
        if row.get("spacing_violation"): reasons.append("spacing violation")
        if row.get("audit_ok") is False: reasons.append("audit incomplete")
        eligible = not reasons
        chosen = eligible and sum(decision.selected for decision in decisions) < limit
        if not chosen and not reasons: reasons.append("lower ranked eligible candidate")
        decisions.append(SelectionDecision(str(row["candidate_id"]), chosen, index + 1 if chosen else None, tuple(reasons)))
    return decisions